## What is happening in the next code?
This code starts the final test evaluation. It loads the final model checkpoint (`resnet18_trainval_final.pt`, trained on train + validation) and creates a `final_test` output folder for the results.
From the checkpoint it reads the saved settings and the class mapping, and builds the sorted list of class names. It also loads the test CSV, the images that were kept untouched until now.
It rebuilds a ResNet18 with Dropout before the final layer, loads the saved weights, moves the model to the CPU, and switches it to eval mode, so dropout is off and predictions are stable.
Then it runs safety checks: the checkpoint is from the last training epoch, the test labels match the class mapping, and no test image path or file hash is repeated.
Finally, it prints the checkpoint name, the final epoch, the number of test images, the class names, and the device. No predictions are made yet.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models import resnet18

PROJECT_ROOT = Path(
    r"D:\Maktab_Sharif_Projects\traffic-vehicle-classification"
)
DEVICE = torch.device("cpu")

CHECKPOINT_PATH = (
    PROJECT_ROOT / "outputs" / "final_model"
    / "resnet18_trainval_final.pt"
)
TEST_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "final_test"
TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

config = checkpoint["config"]
class_to_idx = checkpoint["class_to_idx"]
classes = sorted(class_to_idx, key=class_to_idx.get)

test_df = pd.read_csv(
    PROJECT_ROOT / "data" / "manifests" / "test.csv"
)

model = resnet18(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(p=config["dropout"]),
    nn.Linear(model.fc.in_features, len(classes))
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

assert checkpoint["epoch"] == config["epochs"]
assert set(test_df["label"]) == set(class_to_idx)
assert test_df["path"].is_unique
assert test_df["sha256"].is_unique

print("Checkpoint:", CHECKPOINT_PATH.name)
print("Final training epoch:", checkpoint["epoch"])
print("Test images:", len(test_df))
print("Classes:", classes)
print("Device:", DEVICE)
print("Final model loaded for evaluation.")